# Notebook 03 v2 — CIC-IDS2017 calibration (hybrid Platt/isotonic)

**What this notebook does:**

Fits per-class calibrators on the calibration set carved out in Notebook 01-CIC v2 (M6 fix), applies them to the test set, and reports per-class calibration quality.

**Calibration strategy (hybrid Platt/isotonic, decided in response to TA's rare-class caveat):**

- **Isotonic regression** for classes with n_calib ≥ 30
- **Platt scaling** for classes with n_calib < 30

CIC calibration set sizes:
- Normal: 32,152 → isotonic
- DoS: 5,376 → isotonic
- Probe: 2,248 → isotonic
- R2L: 223 → isotonic
- **U2R: 7 → Platt** (the rare class needing Platt, smaller than NSL's n=10)

**Why CIC U2R is the most stringent test:** n=7 is the smallest calibration class across all three datasets. Platt's 2-parameter fit on n=7 is at the lower bound of statistical feasibility. We report it honestly.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'cic_ids2017_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PROBS_DIR = MODELS_DIR / 'probabilities'   # CIC splits predictions and probabilities like UNSW
CALIB_OUT_DIR = Path(REPO) / 'calibrators' / DATASET
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

PLATT_THRESHOLD = 30
ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Probabilities source: {PROBS_DIR}')
print(f'Output dir: {CALIB_OUT_DIR}')

## 2. Load labels (with defensive schema handling)

In [ ]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)

print(f'class_mappings.json keys: {list(class_info.keys())}')

# CIC may use different key names than NSL/UNSW
mapping_5 = None
for key_candidate in ['multiclass_5', 'five_class_id_to_name', '5class', 'five_class', 'multiclass',
                       'cic_id_to_name', 'class_id_to_name', 'id_to_name']:
    if key_candidate in class_info:
        mapping_5 = class_info[key_candidate]
        print(f"Using key: '{key_candidate}'")
        break

if mapping_5 is None:
    print(f'\nFull JSON content:')
    print(json.dumps(class_info, indent=2))
    raise KeyError('No 5-class mapping found. Update the candidate list.')

INT_TO_CATEGORY = {int(k): v for k, v in mapping_5.items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'\nCalibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print(f'5-class names: {CLASS_NAMES_5}')
print()

# Per-class sizes
calib_counts_5 = Counter(y_calib_5)
calib_counts_b = Counter(y_calib_b)

print('Per-class calibration set sizes:')
print(f'  Binary:')
for c in [0, 1]:
    n = calib_counts_b[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {"Normal" if c == 0 else "Attack":8s}: n={n:>6,}  → {strat}')

print(f'  5-class:')
for c in range(5):
    n = calib_counts_5[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {CLASS_NAMES_5[c]:8s}: n={n:>6,}  → {strat}')

# Class prior comparison for context
test_counts_5 = Counter(y_test_5)
n_calib = len(y_calib_5)
n_test = len(y_test_5)
print()
print('CIC class-prior comparison (for distribution-shift context):')
print(f'  {"Class":<8} {"p_calib (%)":>12} {"p_test (%)":>12} {"ratio":>8}')
for c in range(5):
    p_calib = calib_counts_5[c] / n_calib
    p_test = test_counts_5[c] / n_test
    ratio = p_test / p_calib if p_calib > 0 else float('inf')
    print(f'  {CLASS_NAMES_5[c]:<8} {100*p_calib:>12.2f} {100*p_test:>12.2f} {ratio:>8.2f}x')

## 3. Helper functions (identical to NSL hybrid notebook)

In [ ]:
def fit_calibrator(p_calib, y_indicator, n_class):
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        cal = LogisticRegression(C=1e10, solver='lbfgs')
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (probs >= lo) & (probs <= hi) if i == n_bins - 1 else (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)

def calibrate_model_per_class(p_calib_2d, y_calib, p_test_2d, y_test, n_classes):
    p_test_cal = np.zeros_like(p_test_2d)
    strategies, brier_pre, brier_post, ece_pre, ece_post = {}, {}, {}, {}, {}
    calib_counts = Counter(y_calib)
    
    for c in range(n_classes):
        y_calib_c = (y_calib == c).astype(int)
        y_test_c = (y_test == c).astype(int)
        n_c = calib_counts[c]
        cal, strat = fit_calibrator(p_calib_2d[:, c], y_calib_c, n_c)
        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_2d[:, c])
        strategies[c] = strat
        brier_pre[c] = float(brier_score_loss(y_test_c, p_test_2d[:, c]))
        brier_post[c] = float(brier_score_loss(y_test_c, p_test_cal[:, c]))
        ece_pre[c] = expected_calibration_error(p_test_2d[:, c], y_test_c, ECE_N_BINS)
        ece_post[c] = expected_calibration_error(p_test_cal[:, c], y_test_c, ECE_N_BINS)
    
    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    p_test_cal_renorm = p_test_cal / row_sums
    
    return {
        'p_test_cal': p_test_cal_renorm,
        'strategies': strategies,
        'brier_pre': brier_pre, 'brier_post': brier_post,
        'ece_pre': ece_pre, 'ece_post': ece_post,
    }

def calibrate_binary(p_calib_2d, y_calib, p_test_2d, y_test):
    p_calib_attack = p_calib_2d[:, 1]
    p_test_attack = p_test_2d[:, 1]
    calib_counts = Counter(y_calib)
    n_attack = calib_counts[1]
    cal, strat = fit_calibrator(p_calib_attack, y_calib, n_attack)
    p_test_cal_attack = apply_calibrator(cal, strat, p_test_attack)
    p_test_cal_attack = np.clip(p_test_cal_attack, 0, 1)
    p_test_cal_2d = np.column_stack([1 - p_test_cal_attack, p_test_cal_attack])
    return {
        'p_test_cal': p_test_cal_2d,
        'strategies': {1: strat},
        'brier_pre': {1: float(brier_score_loss(y_test, p_test_attack))},
        'brier_post': {1: float(brier_score_loss(y_test, p_test_cal_attack))},
        'ece_pre': {1: expected_calibration_error(p_test_attack, y_test, ECE_N_BINS)},
        'ece_post': {1: expected_calibration_error(p_test_cal_attack, y_test, ECE_N_BINS)},
    }

print('✓ Helper functions loaded')

## 4. Calibrate all 9 models

In [ ]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task})...')
    p_calib = np.load(PROBS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PROBS_DIR / f'{name}_test_proba.npy')
    
    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])
    
    if task == 'binary':
        result = calibrate_binary(p_calib, y_calib_b, p_test, y_test_b)
    else:
        result = calibrate_model_per_class(p_calib, y_calib_5, p_test, y_test_5, n_classes=5)
    
    np.save(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy', result['p_test_cal'])
    
    print(f'  Strategies: {result["strategies"]}')
    if task == '5class':
        for c in range(5):
            print(f'  {CLASS_NAMES_5[c]:8s}: Brier {result["brier_pre"][c]:.4f} → {result["brier_post"][c]:.4f}'
                  f'   ECE {result["ece_pre"][c]:.4f} → {result["ece_post"][c]:.4f}'
                  f'   [{result["strategies"][c]}]')
    else:
        print(f'  Attack  : Brier {result["brier_pre"][1]:.4f} → {result["brier_post"][1]:.4f}'
              f'   ECE {result["ece_pre"][1]:.4f} → {result["ece_post"][1]:.4f}'
              f'   [{result["strategies"][1]}]')
    
    calibration_summary[name] = {
        'task': task,
        'strategies': {int(k): v for k, v in result['strategies'].items()},
        'brier_pre':  {int(k): v for k, v in result['brier_pre'].items()},
        'brier_post': {int(k): v for k, v in result['brier_post'].items()},
        'ece_pre':  {int(k): v for k, v in result['ece_pre'].items()},
        'ece_post': {int(k): v for k, v in result['ece_post'].items()},
    }

with open(CALIB_OUT_DIR / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary: {CALIB_OUT_DIR}/calibration_summary.json')

## 5. Summary table

In [ ]:
rows = []
for name, task in ALL_MODELS:
    s = calibration_summary[name]
    if task == '5class':
        brier_pre_macro = np.mean(list(s['brier_pre'].values()))
        brier_post_macro = np.mean(list(s['brier_post'].values()))
        ece_pre_macro = np.mean(list(s['ece_pre'].values()))
        ece_post_macro = np.mean(list(s['ece_post'].values()))
        platt_classes = [CLASS_NAMES_5[c] for c, st in s['strategies'].items() if st == 'platt']
        rows.append({
            'Model': name,
            'Task': '5-class',
            'Brier pre (macro)':  round(brier_pre_macro,  5),
            'Brier post (macro)': round(brier_post_macro, 5),
            'ECE pre (macro)':    round(ece_pre_macro,    5),
            'ECE post (macro)':   round(ece_post_macro,   5),
            'Platt for': ','.join(platt_classes) if platt_classes else '—',
        })
    else:
        rows.append({
            'Model': name,
            'Task': 'binary',
            'Brier pre (macro)':  round(s['brier_pre'][1],  5),
            'Brier post (macro)': round(s['brier_post'][1], 5),
            'ECE pre (macro)':    round(s['ece_pre'][1],    5),
            'ECE post (macro)':   round(s['ece_post'][1],   5),
            'Platt for': '—' if s['strategies'][1] == 'isotonic' else 'attack class',
        })

df = pd.DataFrame(rows)
print('\nCIC v2 — Hybrid Platt/Isotonic Calibration Results')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
csv_path = TABLES_DIR / 'cic_v2_calibration_summary.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## 6. Per-class breakdown (5-class models only)

In [ ]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    s = calibration_summary[name]
    for c in range(5):
        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Strategy': s['strategies'][c],
            'Brier pre':  round(s['brier_pre'][c],  5),
            'Brier post': round(s['brier_post'][c], 5),
            'ECE pre':    round(s['ece_pre'][c],    5),
            'ECE post':   round(s['ece_post'][c],   5),
        })

df_perclass = pd.DataFrame(perclass_rows)
print('\nCIC v2 — Per-class calibration breakdown')
print('=' * 120)
print(df_perclass.to_string(index=False))
print('=' * 120)

csv_path = TABLES_DIR / 'cic_v2_calibration_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## 7. Sanity checks

In [ ]:
print('Sanity checks on calibrated probabilities:')
print('-' * 70)
for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5
    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()
    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/03_cic_calibration_v2.ipynb
!git add calibrators/cic_ids2017_v2/
!git add results/tables/cic_v2_calibration_summary.csv
!git add results/tables/cic_v2_calibration_perclass.csv
!git status --short
!git commit -m 'Notebook 03-CIC v2: hybrid Platt/isotonic calibration (U2R n=7 → Platt)'
!git push origin main